In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from dataclasses import dataclass
from tqdm.auto import tqdm
import copy

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

text_english = open("tiny-shakespeare.txt").read()
text_turkish = open("tiny-shakespeare_tr.txt").read()

unique_chars_english = sorted(list(set(text_english)))
unique_chars_turkish = sorted(list(set(text_turkish)))

stoi_en = {ch:i for i, ch in enumerate(unique_chars_english)}
itos_en = {i:ch for i, ch in enumerate(unique_chars_english)}

stoi_tr = {ch:i for i, ch in enumerate(unique_chars_turkish)}
itos_tr = {i:ch for i, ch in enumerate(unique_chars_turkish)}

vocab_size_en = len(stoi_en)
vocab_size_tr = len(stoi_tr)

encode_en = lambda s: [stoi_en[ch] for ch in s]
decode_en = lambda ary: ''.join([itos_en[i] for i in ary])

encode_tr = lambda s: [stoi_tr[ch] for ch in s]
decode_tr = lambda ary: ''.join([itos_tr[i] for i in ary])

encoded_all_data_en = encode_en(text_english)
encoded_all_data_tensor_en = torch.tensor(encoded_all_data_en)
n_en = int(0.9 * len(encoded_all_data_tensor_en))
train_data_tensor_en = encoded_all_data_tensor_en[:n_en]
val_data_tensor_en = encoded_all_data_tensor_en[n_en:]

#send to GPU
train_data_tensor_en = train_data_tensor_en.to(device)
val_data_tensor_en = val_data_tensor_en.to(device)

# encoded_all_data_tr = encode_tr(text_turkish)
# encoded_all_data_tensor_tr = torch.tensor(encoded_all_data_tr)
# n_tr = int(0.9 * len(encoded_all_data_tensor_tr))
# train_data_tensor_tr = encoded_all_data_tensor_tr[:n_tr]
# val_data_tensor_tr = encoded_all_data_tensor_tr[n_tr:]
# train_data_tensor_tr = train_data_tensor_tr.to(device)
# val_data_tensor_tr = val_data_tensor_tr.to(device)

In [ ]:
@dataclass
class Config:
    def __init__(self):
        self.manual_seed: int = 1337
        self.eval_iters: int = 200
        self.max_iters: int = 5000
        self.eval_interval: int = 500
        self.batch_size: int = 64 # = B
        self.block_size: int = 256 #context length = T
        self.learning_rate: float = 3e-4
        self.n_embd: int = 384 # (n_head * head_size) = C, number of channels
        self.n_head: int = 6   
        self.head_size: int = self.n_embd // self.n_head    # (n_embd/n_head): convention
        self.n_layer: int = 6 
        self.dropout: float = 0.2

class AttentionHead(nn.Module):

    def __init__(self, num_embedding_C, head_size_Hs, max_block_size_T, masked=True, dropout=0.2) -> None:
        super().__init__()
        self.num_embedding_C = num_embedding_C
        self.head_size_Hs = head_size_Hs
        self.masked = masked
        # Query can come from a different context
        self.w_query = nn.Linear(in_features=self.num_embedding_C, out_features=self.head_size_Hs, bias=False)
        # Key and Value should come from the same source
        self.w_key = nn.Linear(in_features=self.num_embedding_C, out_features=self.head_size_Hs, bias=False)
        self.w_value = nn.Linear(in_features=self.num_embedding_C, out_features=self.head_size_Hs, bias=False)
        self.tril: torch.Tensor
        self.register_buffer('tril', torch.tril(torch.ones(max_block_size_T, max_block_size_T)))
        self.dropout = nn.Dropout(dropout)


    def forward(self, source_x, target_memory_context=None, key_padding_mask=None):
        if target_memory_context is None:
            Q = self.w_query(source_x) # self attention [B, T_k, C] x [C, Hs] = [B, T_k, Hs]
        else:
            Q = self.w_query(target_memory_context) # cross-attention  # [B, T_q, C] x [C, Hs] = [B, T_q, Hs]

        K = self.w_key(source_x) # [B, T_k, C] x [C, Hs] = [B, T_k, Hs]
        V = self.w_value(source_x) # [B, T_k, Hs]

        dim_k = K.shape[-1] # = Hs
        inv_sqrt_dim_k = (dim_k ** -0.5)
        attention_scores = (Q @ K.transpose(-2, -1)) * inv_sqrt_dim_k # [B, T_q, Hs] @ [B, Hs, T_k] = [B, T_q, T_k]

        if self.masked: # self-attention
            T_q = attention_scores.shape[-2] # "T_kv == T_q" in self attention
            attention_scores = attention_scores.masked_fill(self.tril[:T_q, :T_q] == 0, float('-inf')) # masked attention

        attention_map = F.softmax(attention_scores, dim=-1) #[B, T_q, T_k]
        attention_map = self.dropout(attention_map)

        z = attention_map @ V # [B, T_q, T_k] @ [B, T_k, Hs] = [B, T_q, Hs]

        return z

class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, num_embedding_C, head_size_Hs, max_block_size_T, masked, dropout=0.2) -> None:
        super().__init__()
        self.num_heads = num_heads
        self.attention_heads = nn.ModuleList([AttentionHead(num_embedding_C, head_size_Hs, max_block_size_T, masked, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(in_features=num_heads*head_size_Hs, out_features=num_embedding_C)
        self.dropout = nn.Dropout(dropout)

    def forward(self, source_x, target_memory_context=None, key_padding_mask=None):
        x = torch.cat([h(source_x, target_memory_context, key_padding_mask) for h in self.attention_heads], dim=-1) #cat([B, T, Hs], [B, T, Hs], [B, T, Hs] ...num_heads) cat on Hs, the last column = -1
        out = self.proj(x) # B, T, Hs*num_heads] @ [Hs*num_heads, C] = [B, T, C]
        out = self.dropout(out)
        return out

class FeedForward(nn.Module):

    def __init__(self, num_embedding_C, dropout=0.2) -> None:
        super().__init__()
        # FIX: inner layer must EXPAND by 4x ("d_ff = 2048 for d_model = 512", Vaswani 3.3)
        self.ff = nn.Sequential(nn.Linear(num_embedding_C, 4 * num_embedding_C),
                                nn.ReLU(),
                                nn.Linear(4 * num_embedding_C, num_embedding_C),
                                nn.Dropout(dropout))

    def forward(self, X):
        #Input X from MultiHeadAttention output [B, T, C]
        return self.ff(X)

class DecoderOnlyResidualBlock(nn.Module): #the masked is True for decoder, no information from the future sequences

    def __init__(self, num_heads, num_embedding_C, head_size_Hs, max_block_size_T, dropout=0.2) -> None:
        super().__init__()
        self.mha = MultiHeadAttention(num_heads, num_embedding_C, head_size_Hs, max_block_size_T, True, dropout)
        self.ffwd = FeedForward(num_embedding_C, dropout)
        self.layer_norm1 = nn.LayerNorm(num_embedding_C)
        self.layer_norm2 = nn.LayerNorm(num_embedding_C)

    def forward(self, X):
        #Input is Token_Embeddings + Positional Embeddings [B, T, C]
        x =  X + self.mha(self.layer_norm1(X))
        x =  x + self.ffwd(self.layer_norm2(x))
        return x


class Translator(nn.Module):
    def __init__(self, vocab_size_en, vocab_size_tr, batch_size_B, block_size_T, num_embedding_C, attention_head_size_Hs, num_multi_attention_head_Hn, num_transformer_layer_Nx, dropout = 0.2):
        super().__init__()
        self.vocab_size_en = vocab_size_en
        self.vocab_size_tr = vocab_size_tr
        self.batch_size_B = batch_size_B
        self.block_size_T = block_size_T
        self.num_embedding_C = num_embedding_C
        self.head_size_Hs = attention_head_size_Hs
        self.num_heads = num_multi_attention_head_Hn
        self.num_transformer_layer_Nx = num_transformer_layer_Nx
        self.dropout = dropout

        self.token_embeddings_en = nn.Embedding(num_embeddings=self.vocab_size_en, embedding_dim=self.num_embedding_C)
        self.position_embeddings_en = nn.Embedding(num_embeddings=self.block_size_T, embedding_dim=self.num_embedding_C)
        self.decoder_blocks = nn.Sequential(*[DecoderOnlyResidualBlock(self.num_heads, self.num_embedding_C, self.head_size_Hs, self.block_size_T, self.dropout) for _ in range(self.num_transformer_layer_Nx)])
        self.last_linear_head = nn.Linear(in_features=self.num_embedding_C, out_features=self.vocab_size_en)
        self.last_layernorm_head = nn.LayerNorm(self.num_embedding_C)

        self.apply(self._init_weights)              # FIX: Karpathy's _init_weights

    @staticmethod
    def _init_weights(module):                      # FIX 8
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, X_idx_en, Y_targets = None):
        B, T = X_idx_en.shape # FIX: T comes from the INPUT. No problem in training but might be problem in inference
        
        output_token_embedding = self.token_embeddings_en(X_idx_en) # [B, T, C]
        output_position_embedding = self.position_embeddings_en(torch.arange(T, device=X_idx_en.device)) # [B, T, C]

        x = output_token_embedding + output_position_embedding
        x = self.decoder_blocks(x)
        x = self.last_layernorm_head(x) # [B, T, C] - LayerNorm is shape-preserving. It normalises; it doesn't reduce T dimension.
        logits = self.last_linear_head(x) # [B, T, V = vocab_size]
        
        if Y_targets is None:# when to be used as generator. Only forward no loss calculation
            loss = None
        else:
            B, T, V = logits.shape    
            #flatten logits
            logits = logits.view(B*T, V) #[B, T, V] --> [B*T, V]               
            #flatten targets
            Y_targets = Y_targets.view(B*T)
            #send to loss function
            loss = F.cross_entropy(logits, Y_targets) #[B*T, V] vs [B*T]

        return logits, loss

    # Generate new tokens based on respective last token of a sequence
    # idx = [B, T] where B = 1
    # when B > 1 B times candidate outputs to be generated (on roughly the cost of B=1)
    @torch.no_grad()  # FIX: no graph during sampling
    def generate(self, idx, max_new_tokens): 
        was_training = self.training                # FIX: dropout OFF while sampling
        self.eval()

        for i in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size_T:] # FIX: the MISSING COLON
            # get the output predictions
            logits, _ = self(idx_cond) # forward - output logits = [B, T, V] (if Y_targets is None)
            # get the last tokens of the batches (1 last token if B=1)
            last_token_logit = logits[:, -1, :] #[B=1, V]
            # get the probilities
            probs = F.softmax(last_token_logit, dim=-1) #probilities over "V"
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # [B=1, 1]
            # append to the idx
            idx = torch.cat((idx, idx_next), dim=1)

        self.train(was_training)                   # FIX: restore state
        return idx
  
def getBatch(data, batch_size, block_size):
    ix = torch.randint(0, (len(data)- block_size), (batch_size, ))

    X = torch.stack([data[i:i+block_size] for i in ix ], dim=0) #shifted-right
    Y = torch.stack([data[i+1:i + block_size + 1] for i in ix], dim=0)  #or shifted-left

    X = X.to(device)
    Y = Y.to(device)

    return X, Y

def evaulate_loss(current_model:Translator, cnf:Config):
    was_training = current_model.training # keep previous state
    current_model.eval() #avoid unnecesarry intermediate gradient tensor operations

    out = {}

    with torch.no_grad(): 
        for split in ['train', 'val']:
            temp_losses = torch.zeros(cnf.eval_iters)
            for it in range(cnf.eval_iters):
                if split == 'train':
                    xb, yb = getBatch(train_data_tensor_en, cnf.batch_size, cnf.block_size)
                else:
                    xb, yb = getBatch(val_data_tensor_en, cnf.batch_size, cnf.block_size)

                _, loss = current_model(xb, yb)

                if loss is not None:
                    temp_losses[it] = loss.item()

            out[split] = temp_losses.mean().item()  # FIX: float, not 0-d tensor #mean of all loss of the iterations

    current_model.train(was_training) #restore previous state
    return out

def fit(cnf:Config):
    torch.manual_seed(cnf.manual_seed) #FIX: reproducibility
    translator_model = Translator(vocab_size_en, vocab_size_tr, cnf.batch_size, cnf.block_size, cnf.n_embd, cnf.head_size, cnf.n_head, cnf.n_layer, cnf.dropout)
    translator_model.to(device)

    opt = torch.optim.AdamW(translator_model.parameters(), lr=cnf.learning_rate) #FIX: AdamW is the correct optimizer for transformers, not Adam

    # FIX: state_dict() hands back LIVE tensors. Without a copy these three
    #        names just follow the model and you save the LAST weights, not the best.
    best_state_dict = copy.deepcopy(translator_model.state_dict())
    best_optimizer_dict = copy.deepcopy(opt.state_dict())
    best_val = float('inf')
    for iter in tqdm(range(cnf.max_iters)):

        xb, yb = getBatch(train_data_tensor_en, cnf.batch_size, cnf.block_size) # [B,T]

        _, loss = translator_model(xb, yb)
        opt.zero_grad(set_to_none=True)

        loss.backward()
        opt.step()

        if iter % cnf.eval_interval == 0 or iter == cnf.max_iters - 1:
            losses = evaulate_loss(translator_model, cnf)
            print(f"Current Training step {iter}: loss {loss.item():.4f} Evaluation Step: step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f} ----", flush=True)

            if losses['val'] < best_val:
                best_val = losses['val']
                best_state_dict = copy.deepcopy(translator_model.state_dict())
                best_optimizer_dict = copy.deepcopy(opt.state_dict())    

        # print(f"Current Training step {iter}: loss {loss.item():.4f}", flush=True)

    print(f"Last saved (val {best_val:.4f})")
    torch.save({
                'state_dict': best_state_dict,
                'optimizer':  best_optimizer_dict,   # drop this if you won't resume
                'config':     vars(cnf), # FIX: a dict, not the Config OBJECT
                'stoi':       stoi_en,                     # needed to decode
                'iter':       iter,
                'val_loss':   best_val,
            }, f'gpt_best_{best_val:.4f}.pt')    

def InferenceByTrainedModel(file_name: str):
    ck = torch.load(file_name, weights_only=True, map_location=device)
    cnf = Config()
    
    for k, v in ck['config'].items(): # FIX: rebuild from the dict
        setattr(cnf, k, v)

    model_state_dict = ck['state_dict']

    translator_model = Translator(vocab_size_en, vocab_size_tr, cnf.batch_size, cnf.block_size, cnf.n_embd, cnf.head_size, cnf.n_head, cnf.n_layer, cnf.dropout)
    translator_model.load_state_dict(model_state_dict)
    translator_model.to(device)

    context = torch.zeros((1, 1), dtype=torch.long, device=device)
    print(''.join(decode_en(translator_model.generate(context, max_new_tokens=500)[0].tolist())))


In [ ]:
current_conf = Config()

fit(current_conf)


  0%|          | 0/5000 [00:00<?, ?it/s]

Current Training step 0: loss 4.2183 Evaluation Step: step 0: train loss 3.6600, val loss 3.6878 ----
Current Training step 500: loss 1.8548 Evaluation Step: step 500: train loss 1.7474, val loss 1.8996 ----


In [ ]:
InferenceByTrainedModel("/workspace/repos/jupyter_notebooks/karpathy/gpt_best_158.pt")